In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import Sequential, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import shap
shap.initjs()
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
df = pd.read_csv('/kaggle/input/titanic/train.csv') 
df['Age'] = df['Age'].fillna(df['Age'].median()) 
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0]) 
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1}) 
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True) 
df = df.drop(columns=['Cabin', 'Name', 'Ticket'])  
X = df[['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch','Embarked_S','Embarked_Q']] 
y = df['Survived'] 

In [ ]:
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(X) 
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, 
test_size=0.2, random_state=42) 

In [ ]:
print("Train data shape:",X_train.shape, "Train Label shape", y_train.shape) 
print("Test data shape:",X_test.shape, "Test Label shape", y_test.shape)

In [ ]:
model = Sequential([ 
Input(shape=(X_train.shape[1],)), 
layers.Dense(16, activation='relu'), 
layers.Dense(8, activation='relu'), 
layers.Dense(1, activation='sigmoid')
]) 
model.summary()   


In [ ]:
model.compile(optimizer='adam', 
loss='binary_crossentropy', 
metrics=['accuracy', 
Precision(name='precision'), 
Recall(name='recall')]) 

In [ ]:
early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.plot(history.history['precision'], label='Train Precision')
plt.plot(history.history['val_precision'], label='Val Precision')
plt.plot(history.history['recall'], label='Train Recall')
plt.plot(history.history['val_recall'], label='Val Recall')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Training Progress (Accuracy, Precision, Recall)')
plt.legend()
plt.show()

In [ ]:
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)
print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)

In [ ]:
def predict_fn(X):
    return model.predict(X)

In [ ]:
background = shap.sample(X_train, 100, random_state=42)

In [ ]:
explainer = shap.Explainer(model, background)

In [ ]:
shap_values = explainer(X_test)

In [ ]:
shap.summary_plot(shap_values.values, X_test, feature_names=X_test.columns, plot_type='bar')

In [ ]:
i = 0
shap.plots.force(shap_values[i])

In [ ]:
X_train_np = X_train.values
X_test_np = X_test.values

In [ ]:
lime_explainer = LimeTabularExplainer(
    X_train_np,
    mode='classification',
    training_labels=y_train,
    feature_names=X.columns.tolist(),
    discretize_continuous=True,
    random_state=42
)

In [ ]:
def lime_predict_fn(x):
    preds = model.predict(x)
    return np.hstack((1 - preds, preds))

j = 1
exp = lime_explainer.explain_instance(
    X_test_np[j],
    lime_predict_fn,
    num_features=6
)

In [ ]:
exp.show_in_notebook(show_table=True)

Hello Kerolos, I know you will forget what everything is, so I am Kerolos from last week, hope you are still alive.
now let's remind you of everything you read, let's begin

In the real world we don't just want accurate predictions, we need to understand relationale behid the model, so to ain insights of a ML model we use things like game theory found in shapor by generating local approximations to our predictions found in lime.
let's start with shap, it's an agnostic model (est8afer ellah) the sum of the shap values equal the model prediction minus the baseline.
in Our **first shap graph:**
We are shown feature importance based on average absolute SHAP values. Sex is by far the most important feature followed by Pclass
Question 1: why does sex matter is this a flaw in the dataset or the model?, it doesn't seem important or relative.
**in the second Graph:**
we plot for one passenger with base value of 0.30, with a survival probability of 81% but how does he do this you may ask, it uses adaptive force, it's basicly showing what features push the predicition from a baseline to our 0.81, the base value .3 is the average model output (mean), we reach the 0.81 by adding all the values under the arrows (red +, blue -)
**Third Graph:**
now we get to lime, it's more simple and easy to interpret model we have 1 representing survival and 0 representing death we can see the probabilties and ow each feature effects them still it has the same feture ranking and the same predictions, but how does lime do these things, it takes an instance, taking the feature value and creating a threshold, measuring the imapct when the condition is true or false.

Kerolos Azer Fam 55-13968